# Week 2: T4 + vLLM full sweep

Target: **Kaggle Notebooks** on **T4 x1**.

Extends the Week 1 smoke test into a proper validation set. Measures decode TPS across:
- **Models:** TinyLlama-1.1B, Qwen2.5-1.5B, Qwen2.5-3B (all FP16)
- **Batch sizes:** 1, 4, 16, 32
- **Input lengths:** 128, 512, 2048

Writes `t4_validation.parquet` with columns `model, batch, in_len, out_tokens, elapsed_s, measured_decode_tps, predicted_decode_tps, error_pct`. The Streamlit Validation page reads this file directly.

The point is the **error distribution**: which workloads does the roofline model over- or under-predict, and by how much? If the mean absolute error is >25%, `utilisation_frac` needs recalibrating (probably down for T4, since vLLM on sm_75 is less mature than on Hopper).

In [ ]:
!pip install -q 'vllm==0.6.3' pandas pyarrow 2>&1 | tail -n 3

In [ ]:
import json, time, platform
import torch
import pandas as pd
from vllm import LLM, SamplingParams

env = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'gpu_name': torch.cuda.get_device_name(0),
}
assert 'T4' in env['gpu_name'], f"Expected T4, got {env['gpu_name']}"
print(env)

In [ ]:
MODELS = {
    'TinyLlama-1.1B': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'Qwen2.5-1.5B': 'Qwen/Qwen2.5-1.5B',
    'Qwen2.5-3B': 'Qwen/Qwen2.5-3B',
}
BATCHES = [1, 4, 16, 32]
IN_LENS = [128, 512, 2048]
OUT_TOKENS = 128  # fixed decode length per request
WARMUP = 1

In [ ]:
def _fake_prompt(tokens: int) -> str:
    # A single-token word roughly maps to one tokenizer token for ASCII BPEs.
    # Over-count slightly then let vLLM trim with max_model_len.
    return 'the ' * (tokens * 2)

def measure(llm, batch: int, in_len: int, out_tokens: int = OUT_TOKENS):
    prompts = [_fake_prompt(in_len) for _ in range(batch)]
    params = SamplingParams(temperature=0.0, max_tokens=out_tokens, ignore_eos=True)
    # Warm up
    for _ in range(WARMUP):
        _ = llm.generate(prompts, params, use_tqdm=False)
    t0 = time.perf_counter()
    outs = llm.generate(prompts, params, use_tqdm=False)
    elapsed = time.perf_counter() - t0
    total_out = sum(len(o.outputs[0].token_ids) for o in outs)
    return {'elapsed_s': elapsed, 'out_tokens': total_out, 'measured_decode_tps': total_out / elapsed}

In [ ]:
rows = []
for model_name, hf_id in MODELS.items():
    llm = LLM(model=hf_id, dtype='float16', gpu_memory_utilization=0.85, max_model_len=4096, enforce_eager=True)
    for b in BATCHES:
        for in_len in IN_LENS:
            try:
                m = measure(llm, b, in_len)
                row = {'model': model_name, 'batch': b, 'in_len': in_len, **m}
                rows.append(row)
                print(f'{model_name:<16} B={b:>3} L={in_len:>5}  {m["measured_decode_tps"]:>7.1f} tok/s')
            except Exception as e:
                print(f'{model_name:<16} B={b:>3} L={in_len:>5}  FAILED: {e}')
    del llm
    torch.cuda.empty_cache()

In [ ]:
# Attach predicted values from the roofline model for the same (gpu, model, batch, kv_len)
import sys
sys.path.insert(0, '..')
from core.gpu_specs import load_catalog
from core.cost_engine.kv_cache import TINYLLAMA_1_1B, QWEN2_5_1_5B, QWEN2_5_3B
from core.cost_engine.roofline import decode_tps

ARCHS = {
    'TinyLlama-1.1B': (TINYLLAMA_1_1B, 1.1),
    'Qwen2.5-1.5B': (QWEN2_5_1_5B, 1.5),
    'Qwen2.5-3B': (QWEN2_5_3B, 3.0),
}
t4 = load_catalog().by_id('t4')

for r in rows:
    arch, p = ARCHS[r['model']]
    pred = decode_tps(t4, arch, p, batch_size=r['batch'], kv_seq_len=r['in_len'] + OUT_TOKENS, utilisation_frac=0.55).tokens_per_second
    r['predicted_decode_tps'] = pred
    r['error_pct'] = (pred - r['measured_decode_tps']) / r['measured_decode_tps'] * 100

df = pd.DataFrame(rows)
df.to_parquet('t4_validation.parquet', index=False)
df

In [ ]:
print(f"Mean absolute error: {df['error_pct'].abs().mean():.1f}%")
print(f"Max error: {df['error_pct'].abs().max():.1f}%")
print(df.groupby('model')['error_pct'].agg(['mean', 'std']).round(1))

## What to do with the output

1. Download `t4_validation.parquet` from the Kaggle notebook outputs.
2. Place it at `data/processed/t4_validation.parquet` in the repo and commit.
3. The Streamlit Validation page auto-detects it and renders the predicted-vs-measured scatter + mean absolute error metric.
4. If the mean absolute error is >25%, drop `utilisation_frac` (try 0.5, 0.45) and re-render. The goal is a defensible number — ~15% is a good target on T4 + vLLM.